In [ ]:
import torch

In [ ]:
import re

In [ ]:
import json

In [ ]:
import pickle

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

False

In [ ]:
import numpy as np

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, temperature=0.1)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [ ]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [ ]:
df.head(10)

,title_rus,title_eng,annotation,description
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про..."
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,..."
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...",Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют..."
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...
5,Дизайн учебных материалов (презентаций),Instructional Materials Design (PPT Presentati...,Участникам проекта предлагается оформить две у...,Необходимо оформить две учебные презентации в ...
6,ОРЗ Изучении практик многоязычного обучения в ...,Studying the Practices of Polylingual Educatio...,Участие в нашем проекте будет интересен для те...,Преподавание родного языка и родной литературы...
7,Привязанность к месту и психосоциальное здоров...,Place attachment and psychosocial health: a me...,Привязанность к месту (place attachment) — это...,Привязанность к месту (place attachment) — сра...
8,Подготовка управленческих решений на корпорати...,Preparation of Managerial Decisions at the Co...,Для осуществления управленческих функций необх...,Сложности подготовки управленческих решений на...
9,"Дизайн лонгрида ""Корея в Москве и Санкт-Петерб...","Design of a Longread ""Korea in Moscow and St. ...",Подготовка привлекательного визуального отобра...,Сейчас завершается финальная часть проекта по ...


In [ ]:
with open("artificial_profiles.json", "r") as f:
    profiles = json.load(f)

In [ ]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [ ]:
data = []

for title in titles_with_tags:
  for tag in titles_with_tags[title]:
    data.append((title, tag))

titles_df = pd.DataFrame(data=data, columns=["title", "tag"])

In [ ]:
titles_df.head()

,title,tag
0,Исследование приоритетов и механизмов реализац...,international_relations
1,Исследование приоритетов и механизмов реализац...,political_economics
2,Исследование приоритетов и механизмов реализац...,policy_analysis
3,Исследование приоритетов и механизмов реализац...,BRICS
4,Исследование приоритетов и механизмов реализац...,geopolitics


In [ ]:
results = {}

In [ ]:
BATCH_SIZE = 8
MAX_NEW_TOKENS = 50

for profile, description in tqdm.tqdm(list(profiles.items())):
    SYSTEM_PROMPT = f"""
    You are a {profile}.
    Yours bio: '{description['bio']}'
    Yours tags: '{description['tags']}'
    Evaluate, how much you would like proposed student project based on its title in russian and tags. Rate it on a scale [1, 2, 3, 4, 5], where the higher the better.
    Return stricly JSON.
    For example.
    """
    SYSTEM_PROMPT += """
    Input.
    ```json
    {'title': 'Исследование методов решения задачи линейной регрессии', 'tags': ['machine_learning', 'algorithms']}
    ```
    """
    SYSTEM_PROMPT += """
    Output.
    ```json
    {'score': 5}
    ```
    """

    messages = [
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps({"title": row[1]["title"], "tags": titles_with_tags[row[1]["title"]]}, ensure_ascii=False)},
        )
        for row in titles_df[titles_df["tag"].isin(description["tags"])].iterrows()
    ]

    titles = [msg[1]["content"] for msg in messages]

    scores = {}

    for batch_start in range(0, len(messages), BATCH_SIZE):
        batch = messages[batch_start : batch_start + BATCH_SIZE]

        texts = [
            processor.apply_chat_template(
                msg, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            for msg in batch
        ]

        inputs = processor(text=texts, return_tensors="pt", padding=True).to(
            model.device
        )
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=processor.tokenizer.pad_token_id,
            )

        responses = processor.batch_decode(
            outputs[:, input_len:], skip_special_tokens=False
        )

        for title, response in zip(
            titles[batch_start : batch_start + BATCH_SIZE], responses
        ):
            try:
                json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
                if json_match:
                    json_str = json_match.group(1)
                else:
                    json_str = response
                answer = json.loads(json_str)
            except Exception:
                answer = {"score": None}

            try:
                score = answer.get("score", None)
                score = int(score)
                scores[title] = score
            except (TypeError, ValueError):
                pass

    values = list(scores.values())
    if len(values):
        median = np.median(values)
    else:
        median = np.nan

    print(f"{profile}: {len(values)}, {median}")
    results[profile] = scores

  3%|▎         | 1/31 [00:54<27:21, 54.72s/it]

global_economics_and_geopolitics_analyst: 195, 4.0


  6%|▋         | 2/31 [01:17<17:17, 35.76s/it]

international_relations_and_geopolitics_expert: 85, 4.0


 10%|▉         | 3/31 [01:39<13:49, 29.62s/it]

...

digital_marketing_and_media_strategy: 116, 5.0


 97%|█████████▋| 30/31 [12:23<00:22, 22.97s/it]

media_and_communication_strategist: 45, 5.0


100%|██████████| 31/31 [13:00<00:00, 25.19s/it]

cultural_and_media_anthropologist: 154, 4.0


In [ ]:
with open("artificial_profiles_scores.pkl", "wb") as f:
    pickle.dump(results, f)